# LLM Privacy Gateway — vaultless tokenization for OCSF logs

Sanitizes OCSF security logs so they can be sent to a language model, and
turns the model's answer back into real values. Nothing readable leaves, and
a token the model invents is refused rather than decrypted into something
plausible.

```
Data/ocsf_edr_mock.ndjson    OCSF, one document per line
      |
      +- Stage 0  validate_ocsf       shape checks, before anything trusts it
      +- Stage 1  classify_and_mark   HOST_DESKTOP / USER_PRIV / FILE_PATH_* / AGENT
      +- Stage 2  mark_cii            everything else the field rules know
      +- Stage 3  tokenize_log        FF3 over the suffixes + issued ledger
      v
Data/ocsf_sanitized.log      ready for a model
      |
      +- restore_from_llm -- detokenize -- unmark_cii --> the original values
```

Everything lives in this one notebook, separated by cell tag:

| Tag | Cells | What |
| --- | --- | --- |
| `library` | 5 | Stage 3, Stage 2, Stage 1, the gateway, Stage 0 + the file pipeline |
| `test` | 15 | One pytest module, split by theme |
| `runner` | 1 | Stitches the `test` cells together and runs real pytest |

The corpus is already OCSF, so there is no parsing stage: one line is one
document. Keep scratch work in the untagged cell at the end.

The stages are numbered by when they were built, not by the order they run;
the library cells are ordered by dependency, which is why Stage 3 comes first.

## Why the suffix encodes the value instead of an index

`[INT_IP_192168010087]` rather than `[INT_IP_01]`. An index would need a
global counter and a stored `placeholder → value` table — a plaintext vault,
and a lock on the ingest path. Encoding the value makes Stage 2 a pure
function: the same address yields the same placeholder in every process,
forever, with nothing stored. Reversal needs the FF3 key and nothing else.

The cost is longer tokens, which cost more to send to a model. Correlation is
unaffected: a token repeated five times is still visibly the same entity.

## Quick start

```python
conn = connect_database()
ensure_schema(conn)
key_ring = load_key_ring()

# A whole corpus, with a leak scan of the file that was actually written.
output_path, raw_values = run_pipeline(key_ring=key_ring, conn=conn, limit=1000)

# Or one document at a time.
llm_ready = sanitize_document(raw_ocsf_log, key_ring, conn)
answer    = restore_from_llm(model_response, conn, key_ring, actor="analyst")
```

`run_pipeline` refuses to report success if any raw hostname, internal
address, username or file directory from the inputs survived into the output.
Its scan ignores letter case, because an agent uid spells its host lowercase
(`sangfor-agent-pc-hr-02`) while `device.hostname` spells it uppercase.

## Environment

| Variable | Required | Purpose |
| --- | --- | --- |
| `FF3_KEY` / `FF3_TWEAK` | yes (or the versioned form) | A single key, named by `FF3_KEY_VERSION` |
| `FF3_KEY_<V>` / `FF3_TWEAK_<V>` | for a key ring | One pair per version, e.g. `FF3_KEY_V2` |
| `FF3_ACTIVE_KEY_VERSION` | no | Which version encrypts new tokens |
| `DATABASE_URL` | for DB use | psycopg connection string |

A **14-hex-character (56-bit)** tweak selects FF3-1, the revision NIST kept
after the attacks on the original FF3. A 16-character tweak selects the
original FF3. Prefer 14.

`pip install ff3 psycopg[binary] pytest` — `psycopg` is imported lazily, so
everything except `connect_database()` works without it installed.

## Key rotation

Every token records the key version that encrypted it, and the ring keeps
every version that can still decrypt, so rotation is additive and old tokens
keep working:

```bash
export FF3_KEY_V2=... FF3_TWEAK_V2=...
export FF3_ACTIVE_KEY_VERSION=v2      # new tokens use v2; v1 still decrypts
```

`key_versions_in_use(conn)` counts tokens per version, so a key comes off the
ring when its count is zero. Rotation does not re-encrypt what already
exists; that is a data migration, deliberately outside this module.

## What is protected, and what is not

Redacted: hostname (sub-classified desktop/server), internal and external
IPv4, username (privileged or not), agent uid, log collector, file directory,
MAC, device id, email, network interface, firewall policy name, policy UUID,
URL host.

Deliberately left readable, because an analyst and a model need them:
severity, action, disposition, ports, protocol, timestamps, event codes,
product and vendor, finding title, process names, file name and hash, malware
family.

**Known limits**

- Tokens are global and deterministic. The same host is the same token
  everywhere, which is what makes correlation work and equally what lets two
  datasets be linked. There is no per-tenant separation.
- Small value domains are enumerable. Anyone who can call `sanitize_for_llm`
  can build a lookup table without ever seeing the key; access to the
  gateway is the real control.
- The prefix is cleartext, so the *category* of a value is visible.
- `suffix_len` in the vault leaks the length of the original value.
- `actor` is recorded, not authenticated.
- Stage 2 only redacts what the rules and patterns describe. New log sources
  need their field paths added; `mark_cii` will not guess.
- The compact codec holds **28 characters**. A deeper directory or a longer
  agent uid raises `CiiValueTooLongError` rather than truncating; lifting it
  needs a wider tokenizer alphabet.
- Non-ASCII values raise `NonAsciiValueError` and need their own codec.
- `system` is classified as an ordinary user, which is exactly why the
  hardcoded privileged-user list belongs in an IAM lookup.

## How a host is classified

Priority order, so a record's own statement always beats a guess:

| # | Source | Heuristic? |
| --- | --- | --- |
| 1 | `device.type` (`Desktop`, `Laptop`, `Server`) | no - the schema says so |
| 2 | `device.type_id` (OCSF 1 = Server, 2 = Desktop, 3 = Laptop) | no |
| 3 | hostname prefix (`PC-`, `SRV-`) | **yes**, and only this one warns |

A Detection Finding from an EDR carries `device.type`; a firewall's Network
Activity record does not, which is why step 3 stays. Only step 3 raises the
temporary-heuristic warning, so a run that never falls back stays quiet.

## Stage 3 - tokenization engine

Format-preserving encryption of placeholder suffixes, plus the vault that makes a token's issuance checkable. Runs first because every later cell builds on its regexes and key ring.

In [33]:
from __future__ import annotations

import os
import re
from typing import TYPE_CHECKING, Any, Iterable, Mapping

from ff3 import FF3Cipher

if TYPE_CHECKING:  # psycopg is only needed to talk to a real database
    import psycopg

# The pad character must be part of the cipher alphabet but must never appear
# in a real suffix, otherwise padding is not reversible: rjust("01", 4, "_")
# and the literal suffix "__01" would encrypt to the very same token.
PAD_CHAR = "_"
ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789" + PAD_CHAR

# Suffixes are alphanumeric only, so they can never collide with padding.
PLACEHOLDER_RE = re.compile(r"\[([A-Za-z]+(?:_[A-Za-z]+)*)_([A-Za-z0-9]+)\]")

# Ciphertext may legitimately contain PAD_CHAR, so a token is not always
# shaped like a placeholder. Detokenization scans this broader pattern and
# lets the vault decide what is really a token.
TOKEN_CANDIDATE_RE = re.compile(r"\[[A-Za-z0-9_]+\]")

DEFAULT_KEY_VERSION = "v1"
KEY_ENV_RE = re.compile(r"^FF3_KEY_([A-Za-z0-9]+)$")
RESERVED_KEY_ENV = {"FF3_KEY_VERSION"}


class TokenHallucinationError(Exception):
    """Raised when a token was never issued by this engine.

    Tokens are global: any token this vault has issued can be detokenized by
    any caller holding the key. The check is existence, not ownership.
    """

    def __init__(self, token: str) -> None:
        self.token = token
        super().__init__(f"Token was never issued: {token!r}")


class TokenLengthError(ValueError):
    """Raised when a suffix does not fit the cipher's length bounds."""


class TokenCollisionError(RuntimeError):
    """Raised when one token string maps to two different plaintexts."""

    def __init__(self, token: str) -> None:
        self.token = token
        super().__init__(
            f"Token {token!r} is already issued with different parameters; "
            "it cannot be decrypted unambiguously"
        )


class UnknownKeyVersionError(RuntimeError):
    """Raised when the key that issued a token is not loaded in the key ring.

    This is the genuine "cannot decrypt" case. A token issued under a key
    version that is merely *older* than the active one decrypts normally, as
    long as that version is still on the ring.
    """

    def __init__(self, version: str, available: tuple[str, ...], token: str | None = None) -> None:
        self.version = version
        self.available = tuple(available)
        self.token = token
        super().__init__(
            f"key version {version!r} is not on the key ring "
            f"(loaded: {', '.join(available) or 'none'}); load that key to "
            "detokenize tokens issued under it"
        )


class VersionedCipher:
    """An FF3 cipher bound to the key version that names it.

    Carrying the two together is what stops a token from being stamped with a
    version that does not describe the key that actually encrypted it.
    """

    __slots__ = ("version", "cipher")

    def __init__(self, version: str, cipher: FF3Cipher) -> None:
        self.version = version.lower()
        self.cipher = cipher

    @property
    def minLen(self) -> int:
        return self.cipher.minLen

    @property
    def maxLen(self) -> int:
        return self.cipher.maxLen

    def encrypt(self, plaintext: str) -> str:
        return self.cipher.encrypt(plaintext)

    def decrypt(self, ciphertext: str) -> str:
        return self.cipher.decrypt(ciphertext)

    def __repr__(self) -> str:  # never render key material
        return f"VersionedCipher(version={self.version!r})"


class KeyRing:
    """Every key version that can still decrypt, plus the one that encrypts.

    Rotation adds a version and moves `active_version` forward; the older
    versions stay on the ring, so tokens issued under them keep decrypting:

        token.key_version -> KeyRing.for_version() -> that version's cipher

    This is the key provider seam. `load_key_ring()` fills it from the
    environment, but a KMS or Vault client can build the same mapping.
    """

    def __init__(self, ciphers: Iterable[VersionedCipher], active_version: str) -> None:
        # Indexed by each cipher's own version, so the name a token is stamped
        # with and the name the ring is searched by cannot drift apart.
        self._ciphers = {cipher.version: cipher for cipher in ciphers}
        active = active_version.lower()
        if active not in self._ciphers:
            raise RuntimeError(
                f"active key version {active_version!r} is not on the key ring "
                f"(loaded: {', '.join(sorted(self._ciphers)) or 'none'})"
            )
        self.active_version = active

    @property
    def active(self) -> VersionedCipher:
        """The cipher that encrypts new tokens."""
        return self._ciphers[self.active_version]

    @property
    def versions(self) -> tuple[str, ...]:
        """Every version that can still be decrypted."""
        return tuple(sorted(self._ciphers))

    def for_version(self, version: str) -> VersionedCipher:
        """Resolve the cipher a token was issued under."""
        try:
            return self._ciphers[version.lower()]
        except KeyError:
            raise UnknownKeyVersionError(version, self.versions) from None

    def __repr__(self) -> str:
        return f"KeyRing(versions={self.versions}, active={self.active_version!r})"


def _key_material(environ: Mapping[str, str]) -> dict[str, tuple[str, str | None]]:
    """Collect {version: (key, tweak)} pairs out of the environment."""
    material: dict[str, tuple[str, str | None]] = {}

    # Unversioned pair, for single-key deployments.
    if environ.get("FF3_KEY"):
        version = (environ.get("FF3_KEY_VERSION") or DEFAULT_KEY_VERSION).lower()
        material[version] = (environ["FF3_KEY"], environ.get("FF3_TWEAK"))

    # Versioned pairs win over the unversioned one for the same version.
    for name, key in environ.items():
        if name in RESERVED_KEY_ENV or not key:
            continue
        match = KEY_ENV_RE.match(name)
        if match is None:
            continue
        suffix = match.group(1)
        material[suffix.lower()] = (key, environ.get(f"FF3_TWEAK_{suffix}"))

    return material


def load_key_ring(environ: Mapping[str, str] | None = None) -> KeyRing:
    """Build the key ring from environment variables.

    | Variable | Meaning |
    | --- | --- |
    | `FF3_KEY` / `FF3_TWEAK` | a single key, named by `FF3_KEY_VERSION` (default `v1`) |
    | `FF3_KEY_<V>` / `FF3_TWEAK_<V>` | one pair per version, e.g. `FF3_KEY_V2` |
    | `FF3_ACTIVE_KEY_VERSION` | which version encrypts new tokens |

    To rotate: add the new pair, point `FF3_ACTIVE_KEY_VERSION` at it, and
    leave the old pair loaded until `key_versions_in_use()` reports nothing
    left under it. Version names are matched case-insensitively.

    A 14-hex-character (56-bit) tweak selects FF3-1, the revision NIST kept
    after the attacks on the original FF3; a 16-character (64-bit) tweak
    selects the original FF3. Prefer the 56-bit form.
    """
    environ = os.environ if environ is None else environ
    material = _key_material(environ)
    if not material:
        raise RuntimeError("FF3_KEY (or FF3_KEY_<VERSION>) environment variable is required")

    ciphers = []
    for version, (key, tweak) in material.items():
        if not tweak:
            raise RuntimeError(f"FF3_TWEAK for key version {version!r} is required")
        ciphers.append(
            VersionedCipher(version, FF3Cipher.withCustomAlphabet(key, tweak, ALPHABET))
        )

    requested = environ.get("FF3_ACTIVE_KEY_VERSION") or environ.get("FF3_KEY_VERSION")
    if requested:
        active = requested
    elif len(ciphers) == 1:
        active = ciphers[0].version
    else:
        active = DEFAULT_KEY_VERSION
    return KeyRing(ciphers, active)


def connect_database() -> "psycopg.Connection":
    database_url = os.environ.get("DATABASE_URL")
    if not database_url:
        raise RuntimeError("DATABASE_URL environment variable is required")
    import psycopg

    return psycopg.connect(database_url)


ISSUED_TOKENS_SCHEMA_SQL = """
CREATE SCHEMA IF NOT EXISTS vault_schema;

CREATE TABLE IF NOT EXISTS vault_schema.issued_tokens (
    id          BIGSERIAL PRIMARY KEY,
    token       TEXT NOT NULL,
    prefix      TEXT NOT NULL,
    suffix_len  INTEGER NOT NULL,
    key_version TEXT NOT NULL,
    issued_at   TIMESTAMPTZ NOT NULL DEFAULT now(),
    UNIQUE (token)
);
"""

DETOKENIZE_LOG_SCHEMA_SQL = """
CREATE SCHEMA IF NOT EXISTS metadata_schema;

CREATE TABLE IF NOT EXISTS metadata_schema.detokenize_log (
    id             BIGSERIAL PRIMARY KEY,
    token          TEXT NOT NULL,
    actor          TEXT NOT NULL,
    outcome        TEXT NOT NULL CHECK (outcome IN ('success', 'rejected')),
    detokenized_at TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE INDEX IF NOT EXISTS detokenize_log_actor_idx
    ON metadata_schema.detokenize_log (actor, detokenized_at DESC);
"""

SCHEMA_SQL = ISSUED_TOKENS_SCHEMA_SQL + DETOKENIZE_LOG_SCHEMA_SQL


def ensure_schema(conn: "psycopg.Connection") -> None:
    """Create both schemas if they are missing; safe to run repeatedly.

    Executed without parameters on purpose: psycopg only allows several
    statements in one execute() over the simple query protocol, which it uses
    when no parameters are passed.
    """
    with conn.cursor() as cursor:
        cursor.execute(SCHEMA_SQL)
    conn.commit()


def lookup_issued_token(
    conn: "psycopg.Connection", token: str
) -> tuple[str, int, str] | None:
    """Return (prefix, suffix_len, key_version) for an issued token, or None.

    The prefix is read back from the vault instead of being re-parsed out of
    the token, because ciphertext can contain PAD_CHAR and would otherwise be
    split in a different place than it was at issuance time.
    """
    with conn.cursor() as cursor:
        cursor.execute(
            """
            SELECT prefix, suffix_len, key_version
            FROM vault_schema.issued_tokens
            WHERE token = %s
            """,
            (token,),
        )
        row = cursor.fetchone()
    return None if row is None else (row[0], int(row[1]), row[2])


def is_token_issued(conn: "psycopg.Connection", token: str) -> bool:
    """Return whether this exact token was ever issued."""
    return lookup_issued_token(conn, token) is not None


def key_versions_in_use(conn: "psycopg.Connection") -> dict[str, int]:
    """Count issued tokens per key version.

    A key can only be taken off the ring once its count here reaches zero,
    so this is what makes retiring a rotated-out key a decision rather than a
    guess.
    """
    with conn.cursor() as cursor:
        cursor.execute(
            "SELECT key_version, count(*) FROM vault_schema.issued_tokens "
            "GROUP BY key_version"
        )
        return {version: int(count) for version, count in cursor.fetchall()}


def record_issued_token(
    conn: "psycopg.Connection",
    token: str,
    prefix: str,
    suffix_len: int,
    key_version: str,
) -> None:
    """Record an issued token without creating duplicate records.

    Tokens are global: the same value always produces the same token, so the
    second and later sightings are no-ops.

    `key_version` is required and comes from the VersionedCipher that did the
    encryption, never from the environment, so a token cannot be stamped with
    a version that did not encrypt it.

    Does not commit: the caller owns the transaction so that a whole log is
    tokenized atomically. Use tokenize_log for that.
    """
    with conn.cursor() as cursor:
        cursor.execute(
            """
            INSERT INTO vault_schema.issued_tokens
                (token, prefix, suffix_len, key_version)
            VALUES (%s, %s, %s, %s)
            ON CONFLICT (token) DO NOTHING
            """,
            (token, prefix, suffix_len, key_version),
        )
        if cursor.rowcount == 1:
            return

    # The row already existed. Normally that is the identical value seen
    # again, but two different plaintexts can in principle produce the same
    # token string; that is unrecoverable, so fail loudly here instead of
    # decrypting the wrong value later.
    if lookup_issued_token(conn, token) != (prefix, suffix_len, key_version):
        raise TokenCollisionError(token)


def record_detokenize_attempt(
    conn: "psycopg.Connection", token: str, actor: str, outcome: str
) -> None:
    """Audit a detokenization attempt, including the ones that were refused."""
    with conn.cursor() as cursor:
        cursor.execute(
            """
            INSERT INTO metadata_schema.detokenize_log
                (token, actor, outcome, detokenized_at)
            VALUES (%s, %s, %s, now())
            """,
            (token, actor, outcome),
        )


def _encrypt_suffix(versioned: VersionedCipher, suffix: str) -> str:
    padded_suffix = suffix.rjust(max(versioned.minLen, len(suffix)), PAD_CHAR)
    if len(padded_suffix) > versioned.maxLen:
        raise TokenLengthError(
            f"suffix of length {len(suffix)} exceeds the cipher maximum of "
            f"{versioned.maxLen} characters"
        )
    return versioned.encrypt(padded_suffix)


def tokenize_field(
    key_ring: KeyRing,
    value: str,
    conn: "psycopg.Connection | None" = None,
) -> str:
    """Encrypt only placeholder suffixes, including placeholders in longer text.

    Always encrypts with the ring's active key and records that version
    alongside the token.

    Passing `conn=None` produces tokens without recording them, which is
    useful for previewing but leaves them undetokenizable; production paths
    should always pass a connection.

    Suffixes are upper-cased to fit the cipher alphabet. Tokenizing an
    already tokenized value encrypts it a second time; run this once per log.

    Does not commit; see tokenize_log.
    """
    versioned = key_ring.active

    def replace_placeholder(match: re.Match[str]) -> str:
        prefix = match.group(1).upper()
        suffix = match.group(2).upper()
        token = f"[{prefix}_{_encrypt_suffix(versioned, suffix)}]"
        if conn is not None:
            record_issued_token(conn, token, prefix, len(suffix), versioned.version)
        return token

    return PLACEHOLDER_RE.sub(replace_placeholder, value)


def walk_and_tokenize(
    node: Any,
    key_ring: KeyRing,
    conn: "psycopg.Connection | None" = None,
) -> Any:
    """Recursively tokenize placeholders while preserving the surrounding JSON."""
    if isinstance(node, dict):
        return {key: walk_and_tokenize(value, key_ring, conn) for key, value in node.items()}
    if isinstance(node, list):
        return [walk_and_tokenize(value, key_ring, conn) for value in node]
    if isinstance(node, str):
        return tokenize_field(key_ring, node, conn)
    return node


def tokenize_log(node: Any, key_ring: KeyRing, conn: "psycopg.Connection") -> Any:
    """Tokenize a complete log in a single transaction.

    One commit per log rather than one per placeholder: a log that fails
    halfway through leaves no partially issued tokens behind.
    """
    result = walk_and_tokenize(node, key_ring, conn)
    conn.commit()
    return result


def _detokenize_token(
    conn: "psycopg.Connection", key_ring: KeyRing, token: str, actor: str
) -> str:
    """Validate issuance, resolve the issuing key, decrypt, and audit.

    Does not commit.
    """
    issued = lookup_issued_token(conn, token)
    if issued is None:
        record_detokenize_attempt(conn, token, actor, "rejected")
        raise TokenHallucinationError(token)

    prefix, suffix_len, key_version = issued
    try:
        # The vault says which key issued this token; the ring supplies it.
        # An older version decrypts exactly as well as the active one.
        versioned = key_ring.for_version(key_version)
    except UnknownKeyVersionError as error:
        error.token = token
        record_detokenize_attempt(conn, token, actor, "rejected")
        raise

    # Token is "[" + prefix + "_" + ciphertext + "]". Slicing by the recorded
    # prefix length makes the split match the one used at issuance time even
    # when the ciphertext itself contains PAD_CHAR.
    ciphertext = token[len(prefix) + 2 : -1]
    original_suffix = versioned.decrypt(ciphertext)[-suffix_len:]
    record_detokenize_attempt(conn, token, actor, "success")
    return f"[{prefix}_{original_suffix}]"


def safe_decrypt(
    conn: "psycopg.Connection", key_ring: KeyRing, token: str, actor: str
) -> str:
    """Validate issuance before decrypting and audit every attempt.

    `actor` is trusted as given: this layer records who the caller claims to
    be, it does not authenticate them. Authorization belongs upstream.
    """
    try:
        return _detokenize_token(conn, key_ring, token, actor)
    finally:
        # Commit in finally so a refused attempt is audited too.
        conn.commit()


def detokenize_field(
    value: str, conn: "psycopg.Connection", key_ring: KeyRing, actor: str
) -> str:
    """Replace every issued token inside a string with its placeholder form.

    Bracketed text that was never issued is left alone unless it is shaped
    like a placeholder, which means something invented it. Does not commit.
    """

    def replace_token(match: re.Match[str]) -> str:
        candidate = match.group(0)
        if is_token_issued(conn, candidate):
            return _detokenize_token(conn, key_ring, candidate, actor)
        if PLACEHOLDER_RE.fullmatch(candidate) is None:
            return candidate  # e.g. "logs[6124]", not a token at all
        record_detokenize_attempt(conn, candidate, actor, "rejected")
        raise TokenHallucinationError(candidate)

    return TOKEN_CANDIDATE_RE.sub(replace_token, value)


def walk_and_detokenize(
    node: Any, conn: "psycopg.Connection", key_ring: KeyRing, actor: str
) -> Any:
    """Recursively detokenize a JSON structure. Does not commit."""
    if isinstance(node, dict):
        return {
            key: walk_and_detokenize(value, conn, key_ring, actor)
            for key, value in node.items()
        }
    if isinstance(node, list):
        return [walk_and_detokenize(value, conn, key_ring, actor) for value in node]
    if isinstance(node, str):
        return detokenize_field(node, conn, key_ring, actor)
    return node


def detokenize_log(
    node: Any, conn: "psycopg.Connection", key_ring: KeyRing, actor: str
) -> Any:
    """Detokenize a complete log, auditing every token it touches.

    Tokens issued under different key versions can appear side by side in one
    log; each is resolved against the version recorded for it.
    """
    try:
        return walk_and_detokenize(node, conn, key_ring, actor)
    finally:
        conn.commit()

## Stage 2 - OCSF CII marker

Finds CII in a raw OCSF document and rewrites it as `[PREFIX_SUFFIX]`. Reversible without any stored mapping, which is what keeps the pipeline vaultless.

> Its output still spells out the values. Only Stage 3 hides them.

In [34]:
"""Stage 2 - OCSF CII marker.

Turns raw CII into `[PREFIX_SUFFIX]` placeholders that Stage 3 can encrypt.

The suffix is a *reversible encoding of the value itself*, not an index, so
this stage needs no mapping table and no counter: the same IP always yields
the same placeholder, in any process, forever. That is what keeps the whole
pipeline vaultless - the original is recoverable from the FF3 key alone.

The price is that Stage 2 output is NOT sanitized. `[INT_IP_192168010087]`
still spells out the address. Only Stage 3 hides it. `mark_cii()` is
therefore an internal step; send `sanitize_for_llm()` output to a model, never
this.
"""

import ipaddress
import re
from typing import Any, Callable, NamedTuple

BASE36_DIGITS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
MAX_SUFFIX_LEN = 36  # FF3 maxLen for a radix-37 alphabet
MAX_TEXT_BYTES = 23  # longest value whose base36 encoding still fits


class CiiValueTooLongError(ValueError):
    """Raised when a value cannot be encoded within the cipher's length bound."""

    def __init__(self, prefix: str, value: str, encoded_len: int) -> None:
        self.prefix = prefix
        self.value = value
        self.encoded_len = encoded_len
        super().__init__(
            f"{prefix} value of {len(value.encode('utf-8'))} bytes encodes to "
            f"{encoded_len} characters, over the {MAX_SUFFIX_LEN}-character "
            "limit; give this field its own codec or exclude it"
        )


# --- reversible codecs -----------------------------------------------------
# Each codec maps a value to [A-Z0-9] and back. The placeholder prefix names
# the codec, which is why two source formats of the same concept (dashed
# lowercase UUID vs bare uppercase hex) need two prefixes, not one.


def _to_base36(number: int) -> str:
    if number == 0:
        return "0"
    out = []
    while number:
        number, remainder = divmod(number, 36)
        out.append(BASE36_DIGITS[remainder])
    return "".join(reversed(out))


def _from_base36(text: str) -> int:
    number = 0
    for char in text:
        number = number * 36 + BASE36_DIGITS.index(char)
    return number


def encode_text(value: str) -> str:
    """Any text -> base36. The 0x01 marker preserves leading bytes exactly."""
    return _to_base36(int.from_bytes(b"\x01" + value.encode("utf-8"), "big"))


def decode_text(suffix: str) -> str:
    number = _from_base36(suffix)
    raw = number.to_bytes((number.bit_length() + 7) // 8, "big")
    return raw[1:].decode("utf-8")


def encode_ipv4(value: str) -> str:
    """Dotted quad -> 12 fixed digits, so every address has one encoding."""
    return "".join(f"{int(octet):03d}" for octet in value.split("."))


def decode_ipv4(suffix: str) -> str:
    return ".".join(str(int(suffix[index : index + 3])) for index in range(0, 12, 3))


def encode_mac(value: str) -> str:
    return value.replace(":", "").replace("-", "").upper()


def decode_mac(suffix: str) -> str:
    return ":".join(suffix[index : index + 2] for index in range(0, 12, 2)).lower()


def encode_uuid_dashed(value: str) -> str:
    """Lowercase dashed UUID -> 32 uppercase hex characters."""
    return value.replace("-", "").upper()


def decode_uuid_dashed(suffix: str) -> str:
    lowered = suffix.lower()
    return (
        f"{lowered[:8]}-{lowered[8:12]}-{lowered[12:16]}-"
        f"{lowered[16:20]}-{lowered[20:]}"
    )


def encode_hex(value: str) -> str:
    """Bare uppercase hex is already in the alphabet; keep it byte for byte."""
    return value.upper()


def decode_hex(suffix: str) -> str:
    return suffix.upper()


class Codec(NamedTuple):
    encode: Callable[[str], str]
    decode: Callable[[str], str]


CODECS: dict[str, Codec] = {
    "text": Codec(encode_text, decode_text),
    "ipv4": Codec(encode_ipv4, decode_ipv4),
    "mac": Codec(encode_mac, decode_mac),
    "uuid": Codec(encode_uuid_dashed, decode_uuid_dashed),
    "hex": Codec(encode_hex, decode_hex),
}

# Which codec each placeholder prefix uses. The prefix travels with the data,
# so this table is the only thing needed to reverse Stage 2.
PREFIX_CODEC: dict[str, str] = {
    "INT_IP": "ipv4",
    "EXT_IP": "ipv4",
    "MAC": "mac",
    "POLICY_UUID": "uuid",
    "POLICY_ID": "hex",
    "HOST": "text",
    "DEVICE": "text",
    "USER": "text",
    "EMAIL": "text",
    "IFACE": "text",
    "POLICY": "text",
    "DOMAIN": "text",
    "URL_HOST": "text",
}


def encode_cii(prefix: str, value: str) -> str:
    """Build the `[PREFIX_SUFFIX]` placeholder for one raw value."""
    codec = CODECS[PREFIX_CODEC[prefix]]
    suffix = codec.encode(value)
    if len(suffix) > MAX_SUFFIX_LEN:
        raise CiiValueTooLongError(prefix, value, len(suffix))
    return f"[{prefix}_{suffix}]"


def decode_cii(placeholder: str) -> str:
    """Reverse one placeholder back to the raw value it stands for."""
    match = PLACEHOLDER_RE.fullmatch(placeholder)
    if match is None:
        raise ValueError(f"not a placeholder: {placeholder!r}")
    prefix, suffix = match.group(1).upper(), match.group(2).upper()
    if prefix not in PREFIX_CODEC:
        raise ValueError(f"unknown placeholder prefix: {prefix!r}")
    return CODECS[PREFIX_CODEC[prefix]].decode(suffix)


# --- detection -------------------------------------------------------------

IPV4_RE = re.compile(r"\b\d{1,3}(?:\.\d{1,3}){3}\b")
MAC_RE = re.compile(r"\b(?:[0-9A-Fa-f]{2}[:-]){5}[0-9A-Fa-f]{2}\b")
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
URL_RE = re.compile(r"\bhttps?://[^\s\"'<>,;]+")
UUID_DASHED_RE = re.compile(r"\b[0-9a-f]{8}(?:-[0-9a-f]{4}){3}-[0-9a-f]{12}\b")


def classify_ip(value: str) -> str | None:
    """Internal and external addresses get different prefixes on purpose.

    An analyst still sees "one internal host reached three external hosts"
    without either address being readable.
    """
    try:
        address = ipaddress.ip_address(value)
    except ValueError:
        return None
    if address.version != 4:
        return None
    return "INT_IP" if (address.is_private or address.is_reserved) else "EXT_IP"


# Field paths that hold CII, taken from a survey of the Fortigate and Sangfor
# corpora. List elements are matched as "[]".
FIELD_RULES: dict[str, str] = {
    "src_endpoint.ip": "IP",
    "dst_endpoint.ip": "IP",
    "src_endpoint.mac": "MAC",
    "dst_endpoint.mac": "MAC",
    "src_endpoint.interface_name": "IFACE",
    "dst_endpoint.interface_name": "IFACE",
    "src_endpoint.hostname": "HOST",
    "dst_endpoint.hostname": "HOST",
    "device.name": "HOST",
    "device.hostname": "HOST",
    "device.uid": "DEVICE",
    "actor.user.name": "USER",
    "user.name": "USER",
    "firewall_rule.name": "POLICY",
    "url.hostname": "URL_HOST",
    "unmapped.transip": "IP",
    "unmapped.TranslatedAddress": "IP",
    "unmapped.suser": "USER",
    "unmapped.duser": "USER",
    "unmapped.devname": "HOST",
    "unmapped.devid": "DEVICE",
    "unmapped.poluuid": "POLICY_UUID",
    "unmapped.PolicyUUID": "POLICY_ID",
}

# Fields swept for CII patterns rather than replaced wholesale, because they
# carry the same values again as free text.
FREE_TEXT_PATHS = {"raw_data", "message", "finding_info.desc", "url.url_string"}

# Values that look like CII but carry no information worth protecting.
IGNORED_VALUES = {"", "N/A", "(null)", "null", "-", "::", "0.0.0.0", "unknown"}


class CiiPolicy(NamedTuple):
    """What counts as CII for one deployment."""

    field_rules: dict[str, str] = FIELD_RULES
    free_text_paths: set[str] = FREE_TEXT_PATHS
    ignored_values: set[str] = IGNORED_VALUES
    redact_public_ips: bool = True
    redact_urls: bool = True


DEFAULT_POLICY = CiiPolicy()


def _mark_value(prefix, value, policy, emitted=None):
    """Apply one rule, resolving the IP prefix from the address itself.

    Already-marked values are returned untouched, which makes mark_cii
    idempotent: running it twice cannot double-encode a placeholder.
    """
    if value in policy.ignored_values or PLACEHOLDER_RE.fullmatch(value):
        return value
    if prefix == "IP":
        resolved = classify_ip(value)
        if resolved is None:
            return value
        if resolved == "EXT_IP" and not policy.redact_public_ips:
            return value
        prefix = resolved
    placeholder = encode_cii(prefix, value)
    if emitted is not None:
        emitted.add(placeholder)
    return placeholder


def mark_free_text(text, policy=DEFAULT_POLICY, emitted=None):
    """Replace every CII pattern inside a free-text field.

    This is what stops `raw_data` from leaking values that the structured
    fields already had replaced - both paths encode the same value the same
    way, so one host keeps one identity across the whole document.
    """

    def on_url(match):
        url = match.group(0)
        if not policy.redact_urls:
            return url
        # Keep scheme and path readable; only the host identifies a target.
        parts = re.match(r"(https?://)([^/?#]+)(.*)", url)
        if parts is None:
            return url
        scheme, host, rest = parts.groups()
        host_only = host.split("@")[-1].split(":")[0]
        prefix = classify_ip(host_only) or "URL_HOST"
        if prefix == "EXT_IP" and not policy.redact_public_ips:
            return url
        marked_host = _mark_value(prefix, host_only, policy, emitted)
        return f"{scheme}{host.replace(host_only, marked_host)}{rest}"

    def on_pattern(prefix):
        return lambda match: _mark_value(prefix, match.group(0), policy, emitted)

    # URLs first: their host may itself be an address, and it has to be
    # handled as part of the URL rather than separately.
    text = URL_RE.sub(on_url, text)
    text = EMAIL_RE.sub(on_pattern("EMAIL"), text)
    text = UUID_DASHED_RE.sub(on_pattern("POLICY_UUID"), text)
    text = MAC_RE.sub(on_pattern("MAC"), text)
    return IPV4_RE.sub(on_pattern("IP"), text)


def _mark_fields(node, policy, path, collected, emitted):
    """Pass 1: apply the field rules and remember what each raw value became.

    Free-text fields are left alone here; pass 2 needs the collected values
    before it can touch them.
    """
    if isinstance(node, dict):
        return {
            key: _mark_fields(value, policy, path + (key,), collected, emitted)
            for key, value in node.items()
        }
    if isinstance(node, list):
        return [
            _mark_fields(value, policy, path + ("[]",), collected, emitted)
            for value in node
        ]
    if not isinstance(node, str) or not node:
        return node

    dotted = ".".join(path)
    if dotted in policy.free_text_paths:
        return node
    rule = policy.field_rules.get(dotted)
    if rule is None:
        return node
    marked = _mark_value(rule, node, policy, emitted)
    if marked != node:
        collected[node] = marked
    return marked


def _replace_known_values(text, collected):
    """Replace values already marked elsewhere in the document.

    Fortigate writes `devname="Device-FGT-01"` into raw_data with no pattern
    a regex could recognise, so the only reliable way to catch it is to look
    for the literal values the structured pass already identified. Longest
    first, so a short value cannot eat part of a longer one.
    """
    if not collected:
        return text
    ordered = sorted(collected, key=len, reverse=True)
    pattern = re.compile(
        r"(?<![A-Za-z0-9_])("
        + "|".join(re.escape(value) for value in ordered)
        + r")(?![A-Za-z0-9_])"
    )
    return pattern.sub(lambda match: collected[match.group(1)], text)


def _mark_text(node, policy, path, collected, emitted):
    """Pass 2: sweep the free-text fields."""
    if isinstance(node, dict):
        return {
            key: _mark_text(value, policy, path + (key,), collected, emitted)
            for key, value in node.items()
        }
    if isinstance(node, list):
        return [
            _mark_text(value, policy, path + ("[]",), collected, emitted)
            for value in node
        ]
    if not isinstance(node, str) or not node:
        return node
    if ".".join(path) not in policy.free_text_paths:
        return node
    return mark_free_text(_replace_known_values(node, collected), policy, emitted)


def mark_cii(log, policy=DEFAULT_POLICY, emitted=None):
    """Replace CII in an OCSF document with `[PREFIX_SUFFIX]` placeholders.

    Two passes, because they need each other's results:

    1. **field rules** know the *meaning* of a path, so they pick the right
       prefix - a host name and a device id are both text but are not the
       same thing - and they record every value they replace;
    2. **free-text sweep** puts those same values behind the same
       placeholders wherever they are repeated (`raw_data` restates all of
       them), then runs the pattern matchers to catch addresses, MACs, URLs
       and UUIDs that appear only there.

    One value therefore keeps one identity across the whole document, which
    is what lets a model correlate without ever seeing the value.

    Pass a set as `emitted` to receive every placeholder produced; the
    gateway uses it to prove none of them survived into its output.

    NOT a sanitizer on its own - see this module's docstring.
    """
    collected = {}
    marked = _mark_fields(log, policy, (), collected, emitted)
    return _mark_text(marked, policy, (), collected, emitted)


def unmark_cii(node):
    """Reverse Stage 2: turn placeholders back into the raw values."""
    if isinstance(node, dict):
        return {key: unmark_cii(value) for key, value in node.items()}
    if isinstance(node, list):
        return [unmark_cii(value) for value in node]
    if not isinstance(node, str):
        return node

    def replace(match):
        if match.group(1).upper() not in PREFIX_CODEC:
            return match.group(0)
        try:
            return decode_cii(match.group(0))
        except (ValueError, UnicodeDecodeError):
            return match.group(0)

    return PLACEHOLDER_RE.sub(replace, node)

## Stage 1 - sub-classified prefixes and file paths

`HOST_DESKTOP` / `HOST_SERVER` / `USER_PRIV`, the partial file-path codec, `SRV_LOG_COLLECTOR` and `AGENT`.

> A host's kind comes from `device.type`, then `device.type_id`, and only then from its name. Just that last step is a heuristic, and only it warns.

In [35]:
"""Stage 1 - sub-classified prefixes and the file-path codec.

`mark_cii` picks a prefix from a static path table, so a rule cannot say
"HOST_DESKTOP when the name starts with PC-". Rather than change that logic,
Stage 1 marks those few fields itself with `encode_cii()` - the engine's own
public entry point - and lets `mark_cii` run afterwards over everything else.
That works because `mark_cii` is idempotent: a value that is already a
placeholder is left alone.

Registered into the marker's tables rather than edited into them:
  CODECS["filepath"], CODECS["ascii"]
  PREFIX_CODEC[...]  for the sub-classified, file-path and agent prefixes
  FIELD_RULES[...]   for the paths the EDR corpus uses
"""

import sys
import warnings

# --- a compact codec, because base36-of-UTF-8 does not fit these values ----
# "C:\Users\Public\Downloads" is 25 characters and encodes to 40 base36
# characters through the text codec, over FF3's 36-character maximum. Packing
# printable ASCII at log2(95) bits per character instead of 8 brings it to 32.
ASCII_CHARSET = "".join(chr(code) for code in range(32, 127))
MAX_ASCII_CHARS = 28  # measured: 28 -> 36 base36 characters, 29 overflows


class NonAsciiValueError(ValueError):
    """Raised when a value falls outside the compact codec's character set."""


def encode_ascii(value):
    """Printable-ASCII text -> base36. The leading 1 preserves the first byte."""
    number = 1
    for char in value:
        position = ASCII_CHARSET.find(char)
        if position < 0:
            raise NonAsciiValueError(
                f"{char!r} is outside the printable-ASCII codec; "
                f"give this field its own codec: {value!r}"
            )
        number = number * len(ASCII_CHARSET) + position
    return _to_base36(number)


def decode_ascii(suffix):
    number = _from_base36(suffix)
    out = []
    while number > 1:
        number, remainder = divmod(number, len(ASCII_CHARSET))
        out.append(ASCII_CHARSET[remainder])
    return "".join(reversed(out))


# The directory half of a path is ordinary text once split, so the file-path
# codec is the compact one under the name the placeholder prefix refers to.
def encode_filepath(value):
    return encode_ascii(value)


def decode_filepath(suffix):
    return decode_ascii(suffix)


# --- file paths: encrypt the directory, keep the filename ------------------
# A filename such as update.exe is usually needed to reason about malware and
# rarely identifies a person, while the directory leaks user and share names.
PATH_SEPARATORS = ("\\", "/")

# Ordered: the first match wins, so the more specific pattern comes first.
PATH_HINTS = (
    (("users", "public"), "USER_PUBLIC"),
    (("users",), "USER"),
    (("windows", "system32"), "WINDOWS_SYSTEM"),
    (("windows",), "WINDOWS"),
    (("programdata",), "PROGRAMDATA"),
    (("program files",), "PROGRAM_FILES"),
    (("temp",), "TEMP"),
    (("tmp",), "TEMP"),
    (("appdata",), "APPDATA"),
)

FILE_PATH_PREFIXES = ["FILE_PATH"] + [f"FILE_PATH_{hint}" for _, hint in PATH_HINTS]


def split_file_path(path):
    """Split into (directory, separator, filename) for Windows or POSIX."""
    positions = [path.rfind(separator) for separator in PATH_SEPARATORS]
    cut = max(positions)
    if cut < 0:
        return "", "", path
    return path[:cut], path[cut], path[cut + 1 :]


def path_hint(directory):
    """A short, readable clue about *where* the directory is.

    Derived from the directory itself, so it adds no information the token
    does not already carry - it only makes the placeholder legible to a
    model. Anything unrecognised falls back to the bare FILE_PATH prefix.
    """
    lowered = directory.lower().replace("\\", "/")
    for keywords, hint in PATH_HINTS:
        if all(keyword in lowered for keyword in keywords):
            return hint
    return None


def mark_file_path(path, emitted=None):
    """`C:\\Users\\Public\\Downloads\\update.exe` ->
    `[FILE_PATH_USER_PUBLIC_<encoded>]\\update.exe`"""
    directory, separator, filename = split_file_path(path)
    if not directory:
        return path  # a bare filename has no directory to hide
    if PLACEHOLDER_RE.fullmatch(directory):
        return path  # already marked; marking twice would encode the token
    hint = path_hint(directory)
    prefix = f"FILE_PATH_{hint}" if hint else "FILE_PATH"
    placeholder = encode_cii(prefix, directory)
    if emitted is not None:
        emitted.add(placeholder)
    return f"{placeholder}{separator}{filename}"


# --- sub-classification ----------------------------------------------------
# Priority order for deciding what kind of host a name belongs to:
#
#   1. device.type      - the schema says so outright ("Desktop", "Server")
#   2. device.type_id   - the same statement, numerically
#   3. hostname prefix  - a guess, for events that carry neither
#
# The first two are facts from the record. Only the third is a heuristic, and
# only it warns.
DEVICE_TYPE_PREFIXES = {
    "desktop": "HOST_DESKTOP",
    "laptop": "HOST_DESKTOP",
    "workstation": "HOST_DESKTOP",
    "server": "HOST_SERVER",
}

# OCSF device.type_id, for records that carry the number but not the label.
DEVICE_TYPE_ID_PREFIXES = {
    1: "HOST_SERVER",
    2: "HOST_DESKTOP",
    3: "HOST_DESKTOP",  # Laptop
}

# TEMPORARY, and the only part of this that guesses.
HOSTNAME_PREFIXES = (("PC-", "HOST_DESKTOP"), ("SRV-", "HOST_SERVER"))
PRIVILEGED_USERS = frozenset({"administrator", "root", "admin"})

HEURISTIC_WARNING = (
    "ใช้ heuristic ชั่วคราวสำหรับ desktop/server และ privileged user "
    "classification - ควรแทนที่ด้วย asset inventory / IAM lookup จริงก่อน "
    "production"
)
_warned = False


def reset_heuristic_warning():
    """Let a test observe the warning more than once."""
    global _warned
    _warned = False


def _warn_heuristic():
    global _warned
    if _warned:
        return
    _warned = True
    print(f"WARNING: {HEURISTIC_WARNING}", file=sys.stderr)
    warnings.warn(HEURISTIC_WARNING, stacklevel=2)


def classify_hostname(hostname, device=None):
    """Pick a host prefix, preferring what the record states over a guess.

    `device` is the record's device object when there is one. A Detection
    Finding from an EDR carries `device.type`; a firewall's Network Activity
    record does not, which is why the hostname fallback stays.
    """
    device = device or {}

    stated = str(device.get("type") or "").strip().lower()
    if stated in DEVICE_TYPE_PREFIXES:
        return DEVICE_TYPE_PREFIXES[stated]

    type_id = device.get("type_id")
    if isinstance(type_id, int) and type_id in DEVICE_TYPE_ID_PREFIXES:
        return DEVICE_TYPE_ID_PREFIXES[type_id]

    # Nothing in the record says what this host is, so guess from its name.
    _warn_heuristic()
    upper = hostname.upper()
    for pattern, prefix in HOSTNAME_PREFIXES:
        if upper.startswith(pattern):
            return prefix
    return "HOST"


def classify_username(username, device=None):
    """Names on a hardcoded list are privileged; everyone else is not."""
    _warn_heuristic()
    return "USER_PRIV" if username.lower() in PRIVILEGED_USERS else "USER"


# --- registration into the marker's tables ---------------------------------
def register_extensions():
    """Add this round's codecs and rules to the marker. Idempotent."""
    CODECS["filepath"] = Codec(encode_filepath, decode_filepath)
    CODECS["ascii"] = Codec(encode_ascii, decode_ascii)

    PREFIX_CODEC["HOST_DESKTOP"] = "text"
    PREFIX_CODEC["HOST_SERVER"] = "text"
    PREFIX_CODEC["USER_PRIV"] = "text"
    # The collector's address plus its pid is 24 characters, and an agent uid
    # embeds its host's name; both are past the text codec's 23-byte ceiling.
    PREFIX_CODEC["SRV_LOG_COLLECTOR"] = "ascii"
    PREFIX_CODEC["AGENT"] = "ascii"
    for prefix in FILE_PATH_PREFIXES:
        PREFIX_CODEC[prefix] = "filepath"

    # Paths the marker's own table did not know about.
    FIELD_RULES["device.ip"] = "IP"
    FIELD_RULES["evidences.[].user.name"] = "USER"
    # A Detection Finding nests the user under the process that ran as them,
    # which is a structurally different path from evidences[].user.name.
    FIELD_RULES["evidences.[].process.user.name"] = "USER"
    FIELD_RULES["evidences.[].connection.dst_endpoint.ip"] = "IP"
    FIELD_RULES["device.agent_list.[].uid"] = "AGENT"


register_extensions()

# A command line can name a user or a path, so it is swept like other free
# text rather than kept whole. `raw_data` is already in the marker's own
# free-text set, which is what keeps the CEF copy of every value in step with
# the structured fields.
EDR_POLICY = CiiPolicy(
    free_text_paths=DEFAULT_POLICY.free_text_paths | {"evidences.[].process.cmd_line"}
)


# --- the pre-marking pass --------------------------------------------------
# Only the paths whose prefix depends on the *value or the record*, which is
# the one call `mark_cii` cannot make from its static table. Everything else
# is left to mark_cii, which knows far more paths than these lists do.
#
# Several spellings per concept on purpose: a Detection Finding writes
# `device.hostname` and `evidences[].process.user.name`, a Network Activity
# record writes `device.name` and `unmapped.suser`.
CLASSIFIED_PATHS = {
    "device.hostname": "HOST",
    "device.name": "HOST",
    "src_endpoint.hostname": "HOST",
    "dst_endpoint.hostname": "HOST",
    "evidences.[].user.name": "USER",
    "evidences.[].process.user.name": "USER",
    "actor.user.name": "USER",
    "user.name": "USER",
    "unmapped.suser": "USER",
    "unmapped.duser": "USER",
    "metadata.log_source": "LOG_COLLECTOR",
}

# Paths holding a file path, where only the directory is encrypted.
FILE_PATH_PATHS = frozenset({
    "evidences.[].file.path",
    "file.path",
    "process.file.path",
    "unmapped.filePath",
})


def choose_prefix(kind, value, context):
    """Resolve one CLASSIFIED_PATHS entry to an actual placeholder prefix."""
    if kind == "HOST":
        return classify_hostname(value, context.get("device"))
    if kind == "USER":
        return classify_username(value, context.get("device"))
    if kind == "LOG_COLLECTOR":
        return "SRV_LOG_COLLECTOR"
    raise KeyError(f"no classifier for {kind!r}")


def _classify_node(node, path, collected, emitted, context):
    if isinstance(node, dict):
        return {
            key: _classify_node(value, path + (key,), collected, emitted, context)
            for key, value in node.items()
        }
    if isinstance(node, list):
        return [
            _classify_node(value, path + ("[]",), collected, emitted, context)
            for value in node
        ]
    if not isinstance(node, str) or not node or node in IGNORED_VALUES:
        return node
    if PLACEHOLDER_RE.fullmatch(node):
        return node  # already marked

    dotted = ".".join(path)
    if dotted in FILE_PATH_PATHS:
        marked = mark_file_path(node, emitted)
        if marked != node:
            collected[node] = marked
            # Free text often names the directory without the filename, so
            # record that half too.
            directory, _, _ = split_file_path(node)
            placeholder, _, _ = split_file_path(marked)
            if directory and PLACEHOLDER_RE.fullmatch(placeholder):
                collected[directory] = placeholder
        return marked

    kind = CLASSIFIED_PATHS.get(dotted)
    if kind is None:
        return node
    marked = encode_cii(choose_prefix(kind, node, context), node)
    collected[node] = marked
    if emitted is not None:
        emitted.add(marked)
    return marked


def _sweep_free_text(node, path, collected, policy):
    """Repeat the values marked above wherever free text restates them."""
    if isinstance(node, dict):
        return {
            key: _sweep_free_text(value, path + (key,), collected, policy)
            for key, value in node.items()
        }
    if isinstance(node, list):
        return [
            _sweep_free_text(value, path + ("[]",), collected, policy) for value in node
        ]
    if isinstance(node, str) and ".".join(path) in policy.free_text_paths:
        return _replace_known_values(node, collected)
    return node


def classify_and_mark(document, emitted=None, policy=EDR_POLICY):
    """Mark the sub-classified fields, then repeat those values in free text.

    Returns (document, collected) where `collected` maps each raw value to
    the placeholder that replaced it - the same contract mark_cii keeps
    internally, so `raw_data` and `finding_info.desc` stay consistent with
    the structured fields. The input document is not mutated.
    """
    context = {
        "device": (document.get("device") or {}) if isinstance(document, dict) else {}
    }
    collected = {}
    marked = _classify_node(document, (), collected, emitted, context)
    return _sweep_free_text(marked, (), collected, policy), collected

## The gateway

Stage 2 and Stage 3 wired together, with a check that nothing half-processed can leave.

In [36]:
"""End-to-end LLM privacy gateway.

    raw OCSF  --mark_cii-->  [PREFIX_rawvalue]  --tokenize-->  [PREFIX_cipher]
                             (Stage 2)                          (Stage 3)
                             not yet safe                        LLM-ready

Reverse, for the model's answer:

    [PREFIX_cipher]  --detokenize-->  [PREFIX_rawvalue]  --unmark-->  raw OCSF
"""

import json
from typing import Any, Iterable, Iterator


class UnsanitizedOutputError(RuntimeError):
    """Raised when a document that still spells out CII is about to escape.

    Stage 2 placeholders carry the value in the clear; only Stage 3 hides it.
    Emitting a half-processed document would be the exact failure this
    gateway exists to prevent, so the pipeline checks before returning.
    """

    def __init__(self, leaked: list[str]) -> None:
        self.leaked = leaked
        super().__init__(
            f"{len(leaked)} placeholder(s) were never tokenized and still "
            f"contain their original value, e.g. {leaked[:3]}"
        )


def _surviving_placeholders(document: Any, emitted: set) -> list[str]:
    """Which of Stage 2's placeholders are still present after Stage 3.

    Compared against the exact set Stage 2 produced rather than inferred
    from the text: FF3 preserves both length and alphabet, so ciphertext is
    indistinguishable from a plaintext suffix by inspection. For the `hex`
    and `ipv4` codecs it decodes just as cleanly, which makes any
    "does it still decode?" heuristic useless.
    """
    if not emitted:
        return []
    blob = json.dumps(document)
    return sorted(placeholder for placeholder in emitted if placeholder in blob)


def sanitize_for_llm(
    log: Any,
    key_ring: KeyRing,
    conn: "psycopg.Connection",
    policy: CiiPolicy = DEFAULT_POLICY,
    verify: bool = True,
) -> Any:
    """Turn a raw OCSF log into an LLM-safe document.

    Marks CII, encrypts every placeholder suffix with the ring's active key,
    and records each token in the vault in one transaction. With `verify`
    left on, the result is checked against the exact set of placeholders
    Stage 2 produced before it is handed back.
    """
    emitted: set[str] = set()
    marked = mark_cii(log, policy, emitted)
    tokenized = tokenize_log(marked, key_ring, conn)
    if verify:
        survivors = _surviving_placeholders(tokenized, emitted)
        if survivors:
            raise UnsanitizedOutputError(survivors)
    return tokenized


def restore_from_llm(
    node: Any, conn: "psycopg.Connection", key_ring: KeyRing, actor: str
) -> Any:
    """Turn a model's tokenized answer back into real values.

    Every token is checked against the vault first, so a token the model
    invented is refused rather than decrypted into a plausible-looking
    address. Both outcomes are audited against `actor`.
    """
    return unmark_cii(detokenize_log(node, conn, key_ring, actor))


def sanitize_stream(
    lines: Iterable[str],
    key_ring: KeyRing,
    conn: "psycopg.Connection",
    policy: CiiPolicy = DEFAULT_POLICY,
    verify: bool = True,
) -> Iterator[str]:
    """Sanitize NDJSON one record at a time.

    Yields NDJSON, holding one record in memory at a time, so the multi-
    gigabyte corpora can be processed without being loaded whole. `verify`
    costs one JSON serialization per record; turn it off only for a bulk
    backfill whose output is checked some other way.
    """
    for line in lines:
        line = line.strip()
        if not line:
            continue
        record = sanitize_for_llm(json.loads(line), key_ring, conn, policy, verify)
        yield json.dumps(record)

## Stage 0 and the file pipeline

Shape validation, then Stage 1 → Stage 2 → Stage 3 over a whole corpus, then a scan of the file that was actually written for every raw hostname, internal address, username and file directory the inputs contained.

In [37]:
"""The file pipeline: an OCSF corpus in, a sanitized log out.

    Data/ocsf_edr_mock.ndjson     OCSF, one document per line
      -> Stage 0  validate_ocsf      shape checks, before anything trusts it
      -> Stage 1  classify_and_mark  sub-classified prefixes, file paths
      -> Stage 2  mark_cii           everything else the field rules know
      -> Stage 3  tokenize_log       FF3 over the suffixes + issued ledger
    Data/ocsf_sanitized.log       ready for a model

The corpus is already OCSF, so there is no parsing stage: one line is one
document. This is the encrypt side only; the reverse direction lives in the
gateway cell above and there is no validator or model client yet.
"""

import json
import os
import sys
from pathlib import Path


def locate_data_dir(name="Data"):
    """Find the corpus directory from a kernel, a script, or a temp module."""
    override = os.environ.get("OCSF_DATA_DIR")
    if override:
        return Path(override)
    here = Path(__file__).parent if "__file__" in globals() else Path.cwd()
    for root in (here, here.parent, Path.cwd(), Path.cwd().parent):
        candidate = root / name
        if candidate.is_dir():
            return candidate.resolve()
    return Path(name)  # best guess; only matters when a file is actually read


DATA_DIR = locate_data_dir()
OCSF_INPUT = DATA_DIR / "ocsf_edr_mock.ndjson"
SANITIZED_OUTPUT = DATA_DIR / "ocsf_sanitized.log"


class NotOcsfError(ValueError):
    """Raised when an input line is not a JSON document."""


class OcsfValidationError(ValueError):
    """Raised when a document is JSON but not a usable OCSF record."""


class RawCiiLeakError(AssertionError):
    """Raised when a raw value survived into the sanitized output."""

    def __init__(self, findings):
        self.findings = findings
        listed = ", ".join(f"{kind}={value!r}" for kind, value in findings[:5])
        super().__init__(
            f"{len(findings)} raw CII value(s) found in the sanitized output: {listed}"
        )


# Containers whose shape the marker walks; a scalar where a list belongs
# would be silently skipped rather than marked, so it is caught here instead.
OCSF_CONTAINERS = {"device": dict, "metadata": dict, "finding_info": dict,
                   "evidences": list, "observables": list}


def validate_ocsf(document):
    """Stage 0 - check a record is shaped like OCSF before trusting it.

    The connector already delivers parsed OCSF, so there is nothing to
    convert; what is worth doing is refusing a record whose containers are
    the wrong type, because the marker would walk straight past them and the
    output would look clean while carrying raw values.
    """
    if not isinstance(document, dict):
        raise OcsfValidationError(
            f"expected an OCSF object, got {type(document).__name__}"
        )
    if "class_uid" not in document and "class_name" not in document:
        raise OcsfValidationError("record has neither class_uid nor class_name")
    for key, expected in OCSF_CONTAINERS.items():
        value = document.get(key)
        if value is not None and not isinstance(value, expected):
            raise OcsfValidationError(
                f"{key} should be {expected.__name__}, got {type(value).__name__}"
            )
    return document


def parse_ocsf_line(line):
    """One NDJSON line is one OCSF document; validate, do not convert."""
    stripped = line.strip()
    if not stripped.startswith("{"):
        raise NotOcsfError(f"expected an OCSF JSON document: {stripped[:60]!r}")
    return validate_ocsf(json.loads(stripped))


def collect_raw_cii(document):
    """Every raw value the output must not contain, from one record.

    Driven by the same path tables the marker uses, so it covers whatever
    those tables cover rather than one hardcoded document shape. Internal
    addresses are collected from *every* string, free text included, since
    that is where they hide.
    """
    found = {"hostname": set(), "internal_ip": set(), "username": set(),
             "file_directory": set()}

    def visit(node, path):
        if isinstance(node, dict):
            for key, value in node.items():
                visit(value, path + (key,))
            return
        if isinstance(node, list):
            for value in node:
                visit(value, path + ("[]",))
            return
        if not isinstance(node, str) or not node or node in IGNORED_VALUES:
            return

        dotted = ".".join(path)
        kind = CLASSIFIED_PATHS.get(dotted)
        rule = FIELD_RULES.get(dotted)
        if kind == "HOST" or rule == "HOST":
            found["hostname"].add(node)
        if kind == "USER" or rule == "USER":
            found["username"].add(node)
        if dotted in FILE_PATH_PATHS:
            directory, _, _ = split_file_path(node)
            if directory:
                found["file_directory"].add(directory)
        for candidate in IPV4_RE.findall(node):
            if classify_ip(candidate) == "INT_IP":
                found["internal_ip"].add(candidate)

    visit(document, ())
    return found


def find_leaks(text, raw_values):
    """Look for raw values in sanitized text, ignoring letter case.

    Case matters here: an agent uid spells its host lowercase
    ("sangfor-agent-pc-hr-02") while `device.hostname` spells it uppercase,
    so a case-sensitive scan would call that leak clean.
    """
    lowered = text.lower()
    return [
        (kind, value)
        for kind, values in raw_values.items()
        for value in values
        if value.lower() in lowered
    ]


def sanitize_document(document, key_ring, conn, policy=EDR_POLICY):
    """Stage 0 -> Stage 1 -> Stage 2 -> Stage 3 for one parsed record."""
    validate_ocsf(document)
    emitted = set()
    marked, _ = classify_and_mark(document, emitted, policy)
    marked = mark_cii(marked, policy, emitted)
    tokenized = tokenize_log(marked, key_ring, conn)
    survivors = _surviving_placeholders(tokenized, emitted)
    if survivors:
        raise UnsanitizedOutputError(survivors)
    return tokenized


def run_pipeline(
    input_path=None,
    output_path=None,
    key_ring=None,
    conn=None,
    policy=EDR_POLICY,
    limit=None,
    verbose=True,
):
    """Run Stages 1-3 over an OCSF NDJSON file and write the sanitized log.

    `limit` caps the number of records while iterating.

    Scans the finished file for every raw hostname, internal address,
    username and file directory the inputs contained, and refuses to report
    success if any of them survived.
    """
    input_path = Path(input_path or OCSF_INPUT)
    output_path = Path(output_path or SANITIZED_OUTPUT)
    key_ring = key_ring or load_key_ring()
    if conn is None:
        conn = connect_database()
        ensure_schema(conn)

    if verbose:
        print(f"WARNING: {HEURISTIC_WARNING}\n", file=sys.stderr)

    raw_values = {kind: set() for kind in
                  ("hostname", "internal_ip", "username", "file_directory")}
    written = 0
    with input_path.open(encoding="utf-8") as source, \
            output_path.open("w", encoding="utf-8") as destination:
        for line in source:
            if not line.strip():
                continue
            if limit is not None and written >= limit:
                break
            document = parse_ocsf_line(line)
            for kind, values in collect_raw_cii(document).items():
                raw_values[kind] |= values
            destination.write(
                json.dumps(sanitize_document(document, key_ring, conn, policy)) + "\n"
            )
            written += 1

    # Re-read what was actually written, not what we think we wrote.
    findings = []
    with output_path.open(encoding="utf-8") as produced:
        for line in produced:
            findings += find_leaks(line, raw_values)
    if findings:
        raise RawCiiLeakError(sorted(set(findings)))

    if verbose:
        print(f"Stages 1-3 complete: {written} records -> {output_path}")
        print(f"  distinct tokens issued : {len(getattr(conn, 'issued', ()) or ())}")
        for kind, values in sorted(raw_values.items()):
            print(f"  {kind:15} {len(values):4} distinct raw value(s), 0 in output")
    return output_path, raw_values

## Run the pipeline

The tests live in `test_encryp.ipynb`, which executes the `library` cells above.

In [38]:
# Stages 0-3 over Data/ocsf_edr_mock.ndjson -> Data/ocsf_sanitized.log
#
# Needs FF3_KEY / FF3_TWEAK; prefer a 14-hex-character tweak, which selects
# FF3-1. DATABASE_URL is needed too unless you pass your own connection.
# limit= keeps an iteration short; drop it to process the whole corpus.
try:
    key_ring = load_key_ring()
except RuntimeError as error:
    print(f"skipped: {error}")
else:
    output_path, raw_values = run_pipeline(key_ring=key_ring, limit=1000)
    print(output_path.read_text(encoding="utf-8").splitlines()[0][:400])

skipped: FF3_KEY (or FF3_KEY_<VERSION>) environment variable is required


## Scratch